# Swinging the bat

> The reader on the fast decoder: everything from notebooks 100 to 103, together.

Each notebook built one piece:

- **100**, `voice`: Kokoro taken apart. `analyse` decides how to say a sentence, a decoder turns that into audio, and
  `words` says where every word landed.
- **101**, `sonic`: a speed-up that keeps the pitch and can change speed mid-stream.
- **102**, `player`: play, pause at a word, seek, skip, speed and voice, with any decoder.
- **103**, `coreml`: a decoder for the Mac's GPU, a drop-in for the PyTorch one.

Now we plug the last into the third.

In [ ]:
import time

import numpy as np
from fastcore.test import test_close
from IPython.display import Audio

from text_to_speech.coreml import CoreMLDecoder
from text_to_speech.player import Player, controls, render, sentence_spans, session
from text_to_speech.voice import SR, decode

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.15.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.8.0 is the most recent version that has been tested.
/Users/henrydashwood/text-to-speech/.venv/lib/python3.14/site-packages/coremltools/optimize/torch/palettization/fake_palettize.py:82: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
  n_bits (:obj:`int`): Number of palettization bits. There would be :math:`2^{n\_bits}` unique weights in the ``LUT``.


In [ ]:
article = """Reading aloud used to be the ordinary way to read. For most of history, people who could read were outnumbered
by people who listened, and a book was something performed for a room. Silent reading spread slowly, and was
remarked on when it did.

Recorded books began with the phonograph. Edison listed them among its first uses, but a record held only minutes, so
for decades a novel meant a crate of discs. Talking books for blind readers drove the work forward, and when long
playing records and then cassettes arrived, whole books became something you could carry.

Synthetic voices took longer. The early ones were intelligible but tiring, and listeners who depended on them learned
to run them at three or four times normal speed, because a clear robot at speed beats a natural voice that dawdles.
That habit is why speed control matters so much to anyone building a reader.

A good reading voice today has to do three things at once. It has to sound natural for an hour, not a sentence. It
has to keep up when the listener speeds it up. And it has to start the moment someone presses play."""

print(f"{len(article)} characters, {len(sentence_spans(article))} sentences")

1095 characters, 13 sentences


## The fast decoder

`CoreMLDecoder` converted its models in notebook 103 and saved them, so this only loads them.

In [ ]:
coreml = CoreMLDecoder()

## How far ahead can it stay?

The player renders in the background while it plays. At 3x the listener gets through the audio three times as fast,
so rendering has to run comfortably faster than that or the player stalls between sentences. Rendering a sentence is
`analyse` (the front half, always PyTorch) and the decoder:

In [ ]:
spans = sentence_spans(article)
render(article, *spans[0], decoder=coreml)  # Warm up both
render(article, *spans[0])


def render_speed(decoder):
    "Seconds of audio rendered per second of work, over the whole article."
    t, seconds = time.perf_counter(), 0
    for span in spans:
        seconds += len(render(article, *span, decoder=decoder).audio) / SR
    return seconds / (time.perf_counter() - t)


for name, decoder in [("PyTorch on the CPU", decode), ("Core ML on the GPU", coreml)]:
    print(f"{name}: renders {render_speed(decoder):.0f}x faster than real time")

PyTorch on the CPU: renders 8x faster than real time
Core ML on the GPU: renders 14x faster than real time


Both keep up at 3x on this Mac; the GPU leaves far more room, which a phone, several times slower, will need. What the
listener notices first is the wait after pressing play: the first sentence has to be rendered before anything plays.

In [ ]:
def first_sound(decoder):
    "Seconds from making a player to its first sentence being ready to play."
    t = time.perf_counter()
    player = Player(article, decoder=decoder)
    while not player.ready():
        time.sleep(0.005)
    return time.perf_counter() - t


for name, decoder in [("PyTorch on the CPU", decode), ("Core ML on the GPU", coreml)]:
    print(f"{name}: first sound after {first_sound(decoder):.2f}s")

PyTorch on the CPU: first sound after 0.45s
Core ML on the GPU: first sound after 0.24s


## The controls, on the fast decoder

The same script as notebook 102, this time at up to 3x and on the GPU. The position should climb at the speed asked
for, stop for the pause and jump back at the skip:

In [ ]:
player = Player(article, decoder=coreml)
script = {
    0: player.play,
    3: player.pause,
    4: player.play,
    6: lambda: player.set_speed(3),
    11: lambda: player.skip(-8),
    14: lambda: player.set_voice("bf_emma"),
}
recording, heard, positions = session(player, script, 18)
times = np.arange(1, len(positions) + 1) * 480 / SR


def rate(start, end):
    return (np.interp(end, times, positions) - np.interp(start, times, positions)) / (end - start)


test_close(rate(0.5, 2.5), 1, eps=0.05)
test_close(rate(3.5, 4), 0, eps=0.01)
test_close(rate(7, 10.9), 3, eps=0.15)
assert rate(10.95, 11.05) * 0.1 < -7
assert [w for t, w in heard if t > 14.5], "the new voice should carry on reading"
print(" ".join(w for t, w in heard if 6 < t < 8))
Audio(recording, rate=SR)

could read were outnumbered by people who listened and a book was something performed for a room Silent


## Swing

Press play.

In [ ]:
#| eval: false
controls(Player(article, decoder=coreml))

## Where this leaves us

On this Mac the reader does what audio-reader asks of Alex, with a better-sounding voice, running on the GPU. Notebook
104 showed the same decoder runs at about 10x real time on a Pixel's GPU. What's left is the move from notebooks to
apps: the front half (`analyse`) and the decoder on the phone, the player's logic in Swift or Kotlin, and Sonic, which
compiles for both as it is.